# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

pub_date	title	authors	venue	excerpt	citation	url_slug	paper_url	slides_url	code_url	teaser_img
2024-12-06	Towards open-vocabulary semantic segmentation without semantic labels	Heeseong Shin, <strong>Chaehyun Kim</strong>, Sunghwan Hong, Seokju Cho, Anurag Arnab, Paul Hongsuck Seo, and Seungryong Kim.	In Advances in Neural Information Processing Systems (NeurIPS), 2024		Shin, Heeseong, Chaehyun Kim, Sunghwan Hong, Seokju Cho, Anurag Arnab, Paul Hongsuck Seo, and Seungryong Kim. "Towards open-vocabulary semantic segmentation without semantic labels." Advances in Neural Information Processing Systems 37 (2024): 9153-9177. "Towards open-vocabulary semantic segmentation without semantic labels" <i>NeurIPS2024</i>. 1(1).	PixelCLIP	https://arxiv.org/abs/2409.19846	https://cvlab-kaist.github.io/PixelCLIP	https://github.com/cvlab-kaist/PixelCLIP	../images/title_img/PixelCLIP.png
2025-04-08	D<sup>2</sup>USt3R: Enhancing 3D Reconstruction with 4D Pointmaps for Dynamic Scenes	Jisang Han, Honggyu 

## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)

# fill out NaN with empty string
publications = publications.fillna("")

publications


,pub_date,title,authors,venue,excerpt,citation,url_slug,paper_url,slides_url,code_url,teaser_img
0,2024-12-06,Towards open-vocabulary semantic segmentation ...,"Heeseong Shin, <strong>Chaehyun Kim</strong>, ...",In Advances in Neural Information Processing S...,,"Shin, Heeseong, Chaehyun Kim, Sunghwan Hong, S...",PixelCLIP,https://arxiv.org/abs/2409.19846,https://cvlab-kaist.github.io/PixelCLIP,https://github.com/cvlab-kaist/PixelCLIP,../images/title_img/PixelCLIP.png
1,2025-04-08,D<sup>2</sup>USt3R: Enhancing 3D Reconstructio...,"Jisang Han, Honggyu An, Jaewoo Jung, Takuya Na...",In Advances in Neural Information Processing S...,,"Han, Jisang, Honggyu An, Jaewoo Jung, Takuya N...",DDUSt3R,https://arxiv.org/abs/2504.06264,https://cvlab-kaist.github.io/DDUSt3R/,https://github.com/cvlab-kaist/DDUSt3R/,../images/title_img/DDUSt3R.png
2,2025-09-09,Visual Representation Alignment for Multimodal...,"Heeji Yoon, Jaewoo Jung, Junwan Kim, Hyungyu C...","Arxiv, 2025",,"Yoon, Heeji, Jaewoo Jung, Junwan Kim, Hyungyu ...",VIRAL,https://arxiv.org/abs/2509.07979,https://cvlab-kaist.github.io/VIRAL/,https://github.com/cvlab-kaist/VIRAL/,../images/title_img/VIRAL.png
3,2025-09-20,Seg4Diff: Unveiling Open-Vocabulary Segmentati...,"<strong>Chaehyun Kim</strong>, Heeseong Shin, ...",In Advances in Neural Information Processing S...,,Cite here,Seg4Diff,https://arxiv.org/abs/2509.18096,https://cvlab-kaist.github.io/Seg4Diff/,https://github.com/cvlab-kaist/Seg4Diff,../images/title_img/Seg4Diff.png
4,2025-12-03,C3G: Learning Compact 3D Representations with ...,"Honggyu An, Jaewoo Jung, Mungyeom Kim, Sunghwa...","Arxiv, 2025",,"An, Honggyu, Jaewoo Jung, Mungyeom Kim, Sunghw...",C3G,https://arxiv.org/abs/2512.04021,https://cvlab-kaist.github.io/C3G/,https://github.com/cvlab-kaist/C3G,../images/title_img/C3G.png
5,2025-12-23,Repurposing Video Diffusion Transformers for R...,"Soowon Son, Honggyu An, <strong>Chaehyun Kim</...","Arxiv, 2025",,"Son, Soowon, Honggyu An, Chaehyun Kim, Hyunah ...",DiTracker,https://arxiv.org/abs/2512.20606,https://cvlab-kaist.github.io/DiTracker/,https://github.com/cvlab-kaist/DiTracker,../images/title_img/DiTracker.png


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'

    md += "authors: \"" + item.authors + '"\n'
    
    md += """collection: conferences"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"

    md += "\ncodeurl: '" + item.code_url + "'"

    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\ncategory: conferences"

    md += "\n---"
    
    ## Markdown description for individual page
        
    # if len(str(item.excerpt)) > 5:
    #     md += "\n" + html_escape(item.excerpt) + "\n"

    # if len(str(item.slides_url)) > 5:
    #     md += "\n[Download slides here](" + item.slides_url + ")\n" 

    # if len(str(item.paper_url)) > 5:
    #     md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    # md += "\nRecommended citation: " + item.citation
    
    md += f"""
<div style="display: flex; align-items: flex-start; margin-bottom: 1em;">
    <img src="{item.teaser_img}" alt="teaser" style="width:160px; margin-right:20px; border-radius:8px;">
    <div>
        <h2 style="margin-top:0;">{item.title}</h2>
        <div style="color: #777; margin-bottom: 8px;">{item.authors}</div>
        <div style="font-weight: bold; color: #555;">{item.venue}</div>
        <div style="margin-top:10px;">
        <a href="{item.paper_url}" style="display:inline-block; padding:6px 16px; background:#007bff; color:#fff; border-radius:4px; text-decoration:none; margin-right:8px;">Arxiv Link</a>
        <a href="{item.slides_url}" style="display:inline-block; padding:6px 16px; background:#28a745; color:#fff; border-radius:4px; text-decoration:none;">Project Page</a>
        </div>
    </div>
</div>
    """

    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_publications/

2024-12-06-PixelCLIP.md 2025-09-09-VIRAL.md     2025-12-03-C3G.md
2025-04-08-DDUSt3R.md   2025-09-20-Seg4Diff.md  2025-12-23-DiTracker.md
